# Hopper NEAT — Hyperparameter Search Analysis

Sweep: `pop_size` × `generation_limit` × `seed` → 27 runs total.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('results.csv')
df.head()

## 1 — Overall Summary

In [ ]:
df[['best_fitness', 'time_s']].describe().round(2)

## 2 — Descriptive Statistics by `pop_size`

In [ ]:
df.groupby('pop_size')['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## 3 — Descriptive Statistics by `generation_limit`

In [ ]:
df.groupby('generation_limit')['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## 4 — Descriptive Statistics by `seed`

In [ ]:
df.groupby('seed')['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)

## 5 — Descriptive Statistics by (`pop_size`, `generation_limit`)

Mean and std across seeds for each configuration.

In [ ]:
grouped = df.groupby(['pop_size', 'generation_limit'])['best_fitness'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)
grouped

## 6 — Mean Fitness Pivot Table (pop_size × generation_limit)

In [ ]:
pivot = df.pivot_table(
    index='pop_size',
    columns='generation_limit',
    values='best_fitness',
    aggfunc=['mean', 'std']
).round(2)
pivot

## 7 — Mean Runtime Pivot Table (pop_size × generation_limit)

In [ ]:
df.pivot_table(
    index='pop_size',
    columns='generation_limit',
    values='time_s',
    aggfunc='mean'
).round(1)

## 8 — Best Configuration Overall

In [ ]:
best_row = df.loc[df['best_fitness'].idxmax()]
print(f"Best single run:")
print(f"  pop_size         = {int(best_row['pop_size'])}")
print(f"  generation_limit = {int(best_row['generation_limit'])}")
print(f"  seed             = {int(best_row['seed'])}")
print(f"  best_fitness     = {best_row['best_fitness']:.2f}")
print(f"  time_s           = {best_row['time_s']:.1f}")

print(f"\nBest config by mean fitness (across seeds):")
mean_by_config = df.groupby(['pop_size', 'generation_limit'])['best_fitness'].mean()
best_config = mean_by_config.idxmax()
print(f"  pop_size={best_config[0]}, generation_limit={best_config[1]}")
print(f"  mean fitness = {mean_by_config.max():.2f}")

## 9 — Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# --- By pop_size ---
ax = axes[0]
df.boxplot(column='best_fitness', by='pop_size', ax=ax)
ax.set_title('Fitness by pop_size')
ax.set_xlabel('pop_size')
ax.set_ylabel('best_fitness')

# --- By generation_limit ---
ax = axes[1]
df.boxplot(column='best_fitness', by='generation_limit', ax=ax)
ax.set_title('Fitness by generation_limit')
ax.set_xlabel('generation_limit')

# --- By seed ---
ax = axes[2]
df.boxplot(column='best_fitness', by='seed', ax=ax)
ax.set_title('Fitness by seed')
ax.set_xlabel('seed')

plt.suptitle('')
plt.tight_layout()
plt.savefig('fitness_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

mean_pivot = df.pivot_table(index='pop_size', columns='generation_limit', values='best_fitness', aggfunc='mean')
std_pivot  = df.pivot_table(index='pop_size', columns='generation_limit', values='best_fitness', aggfunc='std')

x = np.arange(len(mean_pivot.index))
width = 0.25

for i, gen in enumerate(mean_pivot.columns):
    ax.bar(x + i * width, mean_pivot[gen], width,
           yerr=std_pivot[gen], capsize=4,
           label=f'gen={gen}')

ax.set_xticks(x + width)
ax.set_xticklabels(mean_pivot.index)
ax.set_xlabel('pop_size')
ax.set_ylabel('Mean best_fitness (± std)')
ax.set_title('Mean Fitness by pop_size & generation_limit')
ax.legend()
plt.tight_layout()
plt.savefig('fitness_barplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns

heatmap_data = df.pivot_table(index='pop_size', columns='generation_limit', values='best_fitness', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Mean Fitness Heatmap (pop_size × generation_limit)')
plt.tight_layout()
plt.savefig('fitness_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()